# Step 6: Deployment

Export the fine-tuned model and deploy it for inference.

**What this notebook covers:**
- Merging LoRA weights with base model
- Exporting to GGUF format for llama.cpp
- Quantization options (Q4_K_M, Q8_0, etc.)
- Starting the FastAPI inference server
- Testing the deployed model

# ⚠️ IMPORTANT - READ BEFORE RUNNING

**This notebook will RE-DEPLOY and OVERWRITE existing files.**

## Purpose
This is an **educational walkthrough** demonstrating how the deployment pipeline works. It will:
- Re-export model to GGUF format from scratch
- OVERWRITE `models/sql-llama-8b-gguf/` if it exists
- Restart the inference server

## When to Use This Notebook
✅ **LEARNING**: Understanding how deployment works  
✅ **DEVELOPMENT**: Testing new quantization settings  
✅ **FRESH START**: Starting completely from scratch

## When NOT to Use This Notebook
❌ **EVALUATION**: You have already trained a model and want to evaluate it  
❌ **PRODUCTION**: You want to use existing deployed model  
❌ **COMPARISON**: You want to compare teacher vs student models

## What You Should Run Instead
If you have completed training and want to evaluate your model first, run:
- `notebooks/07_comparison_glm.ipynb` (if you have GLM API)
- `notebooks/07_comparison_anthropic.ipynb` (if you have Anthropic API)

Only run this notebook (06) AFTER you've evaluated your model and decided it's ready for deployment.

See [docs/notebook-guide.md](docs/notebook-guide.md) for complete guidance.

---

# Step 6: Deployment

Export the fine-tuned model and deploy it for inference.

**What this notebook covers:**
- Merging LoRA weights with base model
- Exporting to GGUF format for llama.cpp
- Quantization options (Q4_K_M, Q8_0, etc.)
- Starting the FastAPI inference server
- Testing the deployed model

In [ ]:
import sys
sys.path.insert(0, '..')

import requests
from src.deploy.export import merge_and_export, push_to_hub
from src.deploy.quantize import quantize_gguf, estimate_model_size
from src.deploy.serve import create_app

# Try importing unsloth for model loading (may not be available on all systems)
try:
    from unsloth import FastLanguageModel
    UNSLOTH_AVAILABLE = True
except ImportError:
    UNSLOTH_AVAILABLE = False
    print("Warning: Unsloth not available. Model loading will fail.")

In [ ]:
# Load fine-tuned model
if not UNSLOTH_AVAILABLE:
    raise ImportError("Unsloth is required to load the model. Install with: pip install unsloth")
model, tokenizer = FastLanguageModel.from_pretrained('models/sql-llama-8b-lora')

In [ ]:
# Export to GGUF
merge_and_export(
    model=model,
    tokenizer=tokenizer,
    output_dir='models/sql-llama-8b-gguf',
    format='gguf'
)
print('Exported to GGUF format')

In [ ]:
# Estimate sizes for different quantizations
param_count = 8e9  # 8B parameters
for bits in [4, 5, 8, 16]:
    size_gb = estimate_model_size(param_count, bits)
    print(f'{bits}-bit: ~{size_gb:.1f} GB')

In [ ]:
# Quantize to Q4_K_M (best quality/size tradeoff)
quantize_gguf(
    input_path='models/sql-llama-8b-gguf/model-f16.gguf',
    output_path='models/sql-llama-8b-gguf/model-q4_k_m.gguf',
    quant_type='q4_k_m'
)

In [ ]:
# Test locally with the FastAPI server
# In a terminal, run:
# uvicorn src.deploy.serve:app --reload
#
# Then test:
response = requests.post('http://localhost:8000/generate_sql', json={
    'question': 'Find all employees who earn more than their manager',
    'schema': open('tasks/sql_generation/schemas.sql').read()
})
print(response.json()['sql'])

In [ ]:
# Push to HuggingFace Hub (optional)
# push_to_hub('models/sql-llama-8b-gguf', repo_id='your-username/sql-llama-8b')